# Module 11 Guided Lab: Final Data Checks, Data Dictionary, and ABT Documentation

**BAN 6003: Data Management and Analytics Integration**

This module is the final preparation step before we apply analytics to an ABT.

We are not learning many new Pandas or SQL functions this week. Instead, we are using the skills we already learned to ask:

> Is this ABT ready for modeling, handoff, and business interpretation?

A good ABT should be checked, documented, and understandable to someone other than the person who built it.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Conduct final sanity checks on a prepared ABT.
2. Check missing values, impossible values, duplicate entity rows, and target variable issues.
3. Create a simple data dictionary for an ABT.
4. Write a transformation and integration summary.
5. Prepare a data handoff package.
6. Apply this same process to your semester project ABT.

Starting next module, we will apply analytics and modeling. That means this is the moment to slow down and make sure the ABT is trustworthy.

## Business Scenario

Imagine you are handing your ABT to another analyst or to your future self.

That person needs to understand:

- what one row means
- where the columns came from
- what transformations were applied
- what the target variable means
- what missing values or limitations still exist
- whether the table is safe to use for modeling

If they cannot understand these points, the ABT is not ready for handoff.

## 0. Setup

This lab uses a small demonstration ABT that the notebook builds from `nycflights13`. You do not need to load any other data.

For the demo, we will build a route-level ABT with one row per origin-destination route.

After the lab, apply the same documentation steps to your own project ABT in your project repository. That work is separate from this notebook.

### Package setup note

You can work in GitHub Codespaces or in local VS Code. On the UTSA campus network Codespaces may not connect; use the local setup there (see the Canvas page on local setup).

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are using **local VS Code** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.


In [ ]:
# Codespaces: nycflights13 is installed from requirements.txt.
# Local only: if the import below fails, uncomment and run this line:
# %pip install nycflights13 "setuptools<81"
# (setuptools<81 is needed because nycflights13 uses pkg_resources, which newer setuptools removed.)

import nycflights13
import pandas as pd
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

output_dir = repo_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
flights = nycflights13.flights.copy()
airports = nycflights13.airports.copy()
airlines = nycflights13.airlines.copy()

## 1. Build a Demo ABT

For this demonstration, we will create a route-level ABT.

**Unit of analysis:** one row per origin-destination route.

This ABT is not meant to be perfect. It is a realistic practice table for final checks and documentation.

In [ ]:
route_abt = (
    flights
    .groupby(["origin", "dest"])
    .agg(
        flight_count=("flight", "count"),
        avg_dep_delay=("dep_delay", "mean"),
        avg_arr_delay=("arr_delay", "mean"),
        max_arr_delay=("arr_delay", "max"),
        avg_distance=("distance", "mean"),
        avg_air_time=("air_time", "mean")
    )
    .reset_index()
)

route_abt.head()

Add destination airport name to make the table more understandable.

This is a left merge because the route-level ABT is our main table and we want to preserve its rows.

In [ ]:
dest_airports = airports[["faa", "name", "lat", "lon"]].rename(
    columns={
        "faa": "dest",
        "name": "dest_airport_name",
        "lat": "dest_lat",
        "lon": "dest_lon"
    }
)

route_abt = route_abt.merge(dest_airports, how="left", on="dest")

route_abt.head()

## 2. Final Sanity Check 1: Row Meaning and Shape

Before checking values, first confirm what one row means.

For this ABT:

> One row should represent one origin-destination route.

The key columns should be `origin` and `dest`.

In [ ]:
route_abt.shape

In [ ]:
route_abt[["origin", "dest"]].head()

### Check duplicate entity rows

If one row should represent one route, then each origin-destination pair should appear only once.

In [ ]:
route_abt[["origin", "dest"]].duplicated().sum()

If this number is greater than zero, the ABT has duplicate entity rows and should not be used for modeling until the issue is resolved.

### Your Turn 1

Write one sentence explaining what one row means in this ABT.

Then identify the key column or columns that define one row.

**Your answer:**  
Type your answer here.

## 3. Final Sanity Check 2: Missing Values

Missing values are not always wrong, but they must be understood.

For an ABT, missing values can affect:

- modeling
- interpretation
- business recommendations
- whether a column should be used at all

In [ ]:
missing_summary = route_abt.isna().sum().reset_index()
missing_summary.columns = ["column_name", "missing_count"]
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(route_abt) * 100).round(1)

missing_summary.sort_values("missing_count", ascending=False)

### Questions to ask

For any column with missing values:

- Is this missingness expected?
- Did it come from the raw data?
- Did it come from an integration step?
- Should the column be kept, dropped, or imputed later?
- Should the limitation be documented?

This module is about checking and documenting, not making every dataset perfect.

### Your Turn 2

Identify one column with missing values, or confirm that no important columns have missing values.

Write one short note explaining what you would do next.

**Your answer:**  
Type your answer here.

## 4. Final Sanity Check 3: Impossible or Suspicious Values

Some values may be technically non-missing but still suspicious.

Examples:

- negative flight counts
- negative distances
- impossible average air times
- unusually large delay values

Not every extreme value is wrong. But it should be noticed.

In [ ]:
route_abt.describe()

In [ ]:
# Basic impossible-value checks
checks = {
    "negative_flight_count": (route_abt["flight_count"] < 0).sum(),
    "zero_or_negative_distance": (route_abt["avg_distance"] <= 0).sum(),
    "zero_or_negative_air_time": (route_abt["avg_air_time"] <= 0).sum(),
}

checks

### Check extreme values

Now sort to inspect routes with very high average arrival delay.

In [ ]:
route_abt.sort_values("avg_arr_delay", ascending=False).head(10)

Extreme values do not automatically mean errors. They may be real business signals. But they should be reviewed before modeling.

### Your Turn 3

`describe()` shows that `max_arr_delay` has a minimum of -22 minutes. A negative *maximum* delay looks impossible.

1. In the code cell below, show the routes where `max_arr_delay` is below 0, with their `flight_count`.
2. Write 1-2 sentences: is this an error? What does it tell you about averages and maximums on routes with very few flights?

In [ ]:
# Your Turn 3
# Show the routes with a negative max_arr_delay and their flight_count.
# Hint: filter route_abt with a Boolean condition, then select a few columns.


**Your answer:**  
Type your answer here.

## 5. Final Sanity Check 4: Target Variable Issues

If the ABT will be used for modeling, the target variable must be clearly defined.

For this demo ABT, suppose our target is:

> average arrival delay by route

That means `avg_arr_delay` is the outcome we may want to explain or predict.

In [ ]:
route_abt["avg_arr_delay"].isna().sum()

In [ ]:
route_abt["avg_arr_delay"].describe()

### Questions to ask about the target

- What exactly does the target measure?
- Is the target missing for some rows?
- Is the target available at the time of prediction?
- Are there extreme values that should be reviewed?
- Would this target support the business question?

These questions help prevent modeling mistakes later.

### Which flights does the target cover?

`flight_count` counts every flight record, including cancelled flights. `avg_arr_delay` averages only the flights that have a recorded arrival delay. The cell below compares the two for each route.

In [ ]:
arr_delay_counts = (
    flights.groupby(["origin", "dest"])["arr_delay"].count()
    .rename("flights_with_arr_delay").reset_index()
)
target_coverage = route_abt[["origin", "dest", "flight_count", "avg_arr_delay"]].merge(
    arr_delay_counts, how="left", on=["origin", "dest"]
)
target_coverage["flights_without_arr_delay"] = (
    target_coverage["flight_count"] - target_coverage["flights_with_arr_delay"]
)

print("Flights without an arrival delay, all routes:", target_coverage["flights_without_arr_delay"].sum())
target_coverage.sort_values("flights_without_arr_delay", ascending=False).head()

### Watch for columns built from the same flights

In this demo ABT, `max_arr_delay` is computed from the same `arr_delay` values as the target, and `avg_dep_delay` comes from the same flights (a late departure usually means a late arrival). If a model later predicts `avg_arr_delay` from these columns, it will look accurate for the wrong reason.

This is not an error in the table, but the receiving analyst must know it. Record it in the data dictionary notes and keep it in mind for Module 12.

### Your Turn 4

Write 2-3 sentences about the demo target `avg_arr_delay`:

1. Exactly which flights does it average? Use one route from the table above as an example.
2. Which route has no target value, and why?
3. Which column should a later model not use to predict this target, and why?

**Your answer:**  
Type your answer here.

## 6. Create a Data Dictionary

A data dictionary explains what each column means.

At minimum, include:

- column name
- data type
- description
- source
- transformation or derivation
- notes or concerns

A data dictionary is not busywork. It is what makes your dataset usable by someone else.

In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": route_abt.columns,
    "data_type": [str(route_abt[col].dtype) for col in route_abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": ""
})

data_dictionary

Now fill in several rows manually as examples.

In a real project, you should complete this for every final ABT column.

In [ ]:
example_cols = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]

data_dictionary.loc[data_dictionary["column_name"] == "origin", example_cols] = [
    "Origin airport code for the route", "flights table", "Used as part of route-level grouping key",
    "None identified"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest", example_cols] = [
    "Destination airport code for the route", "flights table", "Used as part of route-level grouping key",
    "Includes one unusual route, EWR to LGA, with a single cancelled flight"
]

data_dictionary.loc[data_dictionary["column_name"] == "flight_count", example_cols] = [
    "Number of flights observed for the route", "flights table", "Count of flight records grouped by origin and destination",
    "Includes cancelled flights; ranges from 1 to 11,262; 10 routes have fewer than 10 flights, so their averages are unstable"
]

data_dictionary.loc[data_dictionary["column_name"] == "avg_arr_delay", example_cols] = [
    "Average arrival delay for flights on the route, in minutes", "flights table", "Mean of arr_delay grouped by origin and destination",
    "Averages only flights with a recorded arrival delay; missing for EWR to LGA (its only flight was cancelled); negative means early on average"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest_airport_name", example_cols] = [
    "Full name of the destination airport", "airports table", "Joined from airports using dest = faa",
    "Missing for 7 routes whose destination (BQN, PSE, SJU, STT) is not in the airports table"
]

data_dictionary


### Your Turn 5

The cell above documented 5 columns (`origin`, `dest`, `flight_count`, `avg_arr_delay`, `dest_airport_name`). Complete the data dictionary for the 6 remaining ABT columns. Each row should have a useful `description`, `source`, and `transformation_or_derivation`. In `notes_or_concerns`, enter a real concern or `None identified` rather than leaving the field blank.

Use the completed dictionary in the handoff-readiness audit later in the notebook.


In [ ]:
# Your Turn 5 - document every remaining ABT column.
# Hint: copy this pattern and change the column name and three text values.
# data_dictionary.loc[
#     data_dictionary["column_name"] == "column_name_here",
#     ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
# ] = ["...", "...", "...", "..."]


## 7. Transformation and Integration Summary

The data dictionary documents columns. The transformation summary documents the process.

A good transformation summary answers:

- What raw data was used?
- What cleaning or filtering was applied?
- What transformations were created?
- What aggregations changed the unit of analysis?
- What tables were joined?
- What checks were performed?
- What assumptions or limitations remain?

### Example Transformation Summary

For this demo ABT:

1. Started with the `flights` table from `nycflights13`.
2. Aggregated flight-level records to one row per origin-destination route.
3. Calculated flight count, average departure delay, average arrival delay, maximum arrival delay, average distance, and average air time.
4. Joined destination airport name and location from the `airports` table using `dest = faa`.
5. Checked duplicate route keys using `origin` and `dest`.
6. Checked missing values across ABT columns.
7. Reviewed basic impossible values and extreme delay values.
8. The target variable for later modeling may be `avg_arr_delay`, but this choice should be justified by the business question.

### Your Turn 6

The example summary says which checks were performed, but not what they found. The receiving analyst needs both.

Add a short **Findings and limitations** list (3-4 bullets) to the demo summary. Use numbers from Sections 2-5 and from Your Turns 3 and 4, for example duplicate keys, missing values, very small routes, and what the target covers.

**Findings and limitations:**  
- Type your first bullet here.

## 8. Prepare a Handoff Package

A complete ABT handoff package usually has five parts:

1. Final ABT file
2. Complete data dictionary
3. Transformation and integration summary
4. Validation checklist
5. README or short handoff note

In this lab you export **four files** to the repository-level `outputs` folder: the ABT, the data dictionary, the validation checklist, and a **handoff audit** (the evidence behind the checklist). The demo transformation summary is in Section 7. Write the summary and README for your own project ABT in your project repository.

### Audit first, then the checklist

- The **handoff audit** is the evidence: one row of counts, such as duplicate keys, missing target values and blank dictionary cells.
- The **validation checklist** is the status summary built from that evidence. Each check is:
  - `Complete`: done, with no problem, or with a known item that only needs documentation (for example, a missing target value explained in the dictionary).
  - `Needs attention`: a problem that must be fixed or explained before handoff.
  - `In progress`: the data dictionary still has blank cells, so documentation items are not finished.
- The **handoff decision** cites the audit numbers.

You build the audit in Section 9. The notebook then builds the checklist from your audit and exports all four files once.

## 9. Offline Assignment - Handoff-Readiness Audit

An analyst on another team will use the route ABT next week and cannot ask you to explain undocumented details. Decide whether the handoff package is ready.

1. Make sure Your Turn 5 is finished: every ABT column has all four documentation fields.
2. Complete the audit below. Replace each `None` with code that computes the number; you can reuse the checks from Sections 2 to 5.
3. Run the next two cells. The first builds the validation checklist from your audit; the second exports all four handoff files.
4. Write the handoff decision. A real extreme delay can remain if it is documented; do not silently remove it.

In [ ]:
# Offline Assignment - complete the TODO values.
# Replace each None below with code that computes the number, for example route_abt[...].some_check().sum().
# Hints:
# - Use .duplicated().sum() on ["origin", "dest"].
# - Select the two key columns before counting missing values.
# - Use Boolean conditions such as (route_abt["flight_count"] <= 0).sum().
# - For documentation, select the four text fields, strip whitespace, compare to "", then sum.
required_documentation_fields = [
    "description", "source", "transformation_or_derivation", "notes_or_concerns"
]

handoff_audit = pd.DataFrame([{
    "abt_rows": len(route_abt),
    "dictionary_rows": len(data_dictionary),
    "dictionary_matches_abt_columns": len(data_dictionary) == len(route_abt.columns),
    "duplicate_route_keys": None,  # TODO
    "missing_key_values": None,  # TODO
    "nonpositive_flight_counts": None,  # TODO
    "nonpositive_average_distance": None,  # TODO
    "nonpositive_average_air_time": None,  # TODO
    "missing_target_values": None,  # TODO: avg_arr_delay
    "undocumented_dictionary_cells": None,  # TODO
}])

handoff_audit


In [ ]:
# Build the validation checklist from your audit. Run this after the audit cell.
if handoff_audit.isna().any(axis=None):
    print("Complete every audit TODO first, then rerun this cell.")
else:
    a = handoff_audit.iloc[0]
    docs_done = bool(a["dictionary_matches_abt_columns"]) and a["undocumented_dictionary_cells"] == 0
    nonpositive = (a["nonpositive_flight_counts"] + a["nonpositive_average_distance"]
                   + a["nonpositive_average_air_time"])
    missing_cells = int(route_abt.isna().sum().sum())

    def status(passed, otherwise="Needs attention"):
        return "Complete" if passed else otherwise

    validation_checklist = pd.DataFrame([
        ["Unit of analysis is clearly defined", "Complete", "One row per origin-destination route"],
        ["Key columns are identified", status(a["missing_key_values"] == 0),
         f"origin + dest; {a['missing_key_values']} missing key values"],
        ["Duplicate entity rows checked", status(a["duplicate_route_keys"] == 0),
         f"{a['duplicate_route_keys']} duplicate origin-dest keys"],
        ["Missing values checked", status(docs_done, "In progress"),
         f"{missing_cells} missing cells; explained in the data dictionary notes"],
        ["Impossible or suspicious values reviewed", status(nonpositive == 0),
         f"{nonpositive} nonpositive flight counts, distances, or air times"],
        ["Target variable reviewed", status(a["missing_target_values"] == 0 or docs_done, "In progress"),
         f"avg_arr_delay; {a['missing_target_values']} missing target values"],
        ["Data dictionary complete", status(docs_done, "In progress"),
         f"{a['dictionary_rows']} dictionary rows for {len(route_abt.columns)} columns; {a['undocumented_dictionary_cells']} blank cells"],
        ["Transformation summary written", "Complete",
         "Demo summary in Section 7, with findings and limitations (Your Turn 6)"],
    ], columns=["check", "status", "notes"])
    display(validation_checklist)

In [ ]:
# Export the four handoff files. Run this after the checklist cell.
if handoff_audit.isna().any(axis=None):
    print("Complete every audit TODO before exporting the handoff files.")
else:
    handoff_files = {
        "route_abt_module11.csv": route_abt,
        "route_abt_data_dictionary_module11.csv": data_dictionary,
        "route_abt_validation_checklist_module11.csv": validation_checklist,
        "route_abt_handoff_audit_module11.csv": handoff_audit,
    }
    for file_name, table in handoff_files.items():
        table.to_csv(output_dir / file_name, index=False)
        print("Saved:", output_dir / file_name)

### Handoff Decision

**Your response here:** Write 5-6 sentences that approve the handoff, approve it with conditions, or hold it for revision. Cite at least four audit results, identify one remaining limitation or target concern, and state what the receiving analyst should know before using the ABT for modeling.


## 10. Apply This to Your Project (after the lab)

This section is not part of the lab grade. Use the same process when you prepare your project ABT for the next milestone:

1. A clear statement of your ABT unit of analysis.
2. A validation checklist.
3. A data dictionary for your ABT.
4. A transformation and integration summary.
5. A clean handoff folder in GitHub.

This work prepares you for the next module, where we begin applying analytics to the ABT.

## 11. Final Reflection

Write 3-4 sentences about your project ABT. Answer from what you know now; you do not need to build anything:

1. What does one row mean, and which columns are the key?
2. What is your likely target, and is it known for every row?
3. Which check from this lab will you run first on your project ABT, and which column will need the clearest dictionary entry?

**Your reflection:**  
Type your answer here.

## 12. Save Your Work

Before submitting:

1. Complete all Your Turn sections and the handoff-readiness audit.
2. Complete the handoff decision and final reflection.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm all four Module 11 files exist in `outputs`.
5. Save the notebook, then commit and push your work.

```bash
git add .
git commit -m "Complete Module 11 ABT documentation lab"
git push
```


## Optional Practice: Diagnose a Mini Handoff Package (Not Graded)

This short practice does **not** count toward your lab grade. Use it if you want another repetition of the Week 11 handoff-readiness workflow before applying the same checks to your project.

A teammate sends a small route ABT and data dictionary. Do not repair the files immediately. First build an evidence-based audit that identifies whether the package is ready for another analyst.


In [ ]:
practice_abt = pd.DataFrame({
    "origin": ["JFK", "JFK", "JFK", "LGA"],
    "dest": ["ORD", "ATL", "ATL", "BOS"],
    "flight_count": [820, 760, 760, 640],
    "avg_arr_delay": [8.4, 11.2, 11.2, pd.NA],
})

practice_dictionary = pd.DataFrame({
    "column_name": ["origin", "dest", "flight_count", "avg_arr_delay"],
    "description": ["Origin airport code", "Destination airport code", "Number of flights", ""],
    "source": ["flights.origin", "flights.dest", "flights", "flights.arr_delay"],
    "transformation_or_derivation": ["Grouped field", "Grouped field", "COUNT(*)", "AVG(arr_delay)"],
    "notes_or_concerns": ["Part of route key", "Part of route key", "Should be positive", "May be missing"],
})

display(practice_abt)
display(practice_dictionary)


### Practice Task

Create a one-row DataFrame named `practice_handoff_audit` containing:

1. Number of ABT rows.
2. Number of duplicate `origin`-`dest` keys.
3. Number of missing key values.
4. Number of missing `avg_arr_delay` target values.
5. Whether the dictionary has one row for every ABT column.
6. Total blank cells across `description`, `source`, `transformation_or_derivation`, and `notes_or_concerns`.

Hints:

- Use `.duplicated(["origin", "dest"]).sum()` for the composite key.
- Select both key columns before applying `.isna().sum().sum()`.
- For documentation blanks, select the four text fields, apply `.fillna("")`, convert to string, strip whitespace, compare with `""`, and sum twice.
- Audit first. A handoff decision should describe the evidence before recommending repairs.


In [ ]:
# Optional practice workspace
# Replace None with a one-row DataFrame containing the six requested checks.
practice_handoff_audit = None


In [ ]:
# Run this self-check after you create practice_handoff_audit.
if practice_handoff_audit is None:
    print("Complete the optional practice first, then rerun this self-check.")
else:
    expected_practice_audit = pd.DataFrame([{
        "abt_rows": 4,
        "duplicate_route_keys": 1,
        "missing_key_values": 0,
        "missing_target_values": 1,
        "dictionary_matches_abt_columns": True,
        "undocumented_dictionary_cells": 1,
    }])
    print("Your audit:")
    display(practice_handoff_audit)
    print("Expected audit:")
    display(expected_practice_audit)


### Think About the Result

This package should be held for revision: it contains a duplicate route key, a missing target value, and an undocumented dictionary cell. Notice that the dictionary row count alone passes, but completeness still fails. A useful handoff decision should name each issue, explain why it matters to the receiving analyst, and request a specific correction or documented exception.
